# Granite 4.0 1B Base — Blind Spot Evaluation\nColab-ready notebook for reproducing the blind-spot run.

In [ ]:
!pip -q install transformers torch huggingface_hub

In [ ]:
import os, json, re, time\nimport torch\nfrom transformers import AutoTokenizer, AutoModelForCausalLM

In [ ]:
# Optional: set your HF token in Colab secrets and uncomment\n# os.environ['HF_TOKEN'] = 'hf_xxx'

In [ ]:
model_id = 'ibm-granite/granite-4.0-1b-base'\ndevice = 'cuda' if torch.cuda.is_available() else 'cpu'\ntok = AutoTokenizer.from_pretrained(model_id)\nif tok.pad_token_id is None and tok.eos_token_id is not None:\n    tok.pad_token = tok.eos_token\nmodel = AutoModelForCausalLM.from_pretrained(model_id).to(device).eval()\nprint('Loaded', model_id, 'on', device)

In [ ]:
prompts = [\n  {'id':'Q1','input':"Use van't Hoff equation. For N2O4(g) ⇌ 2NO2(g), K1=0.30 at 298 K and ΔH°=+57.2 kJ/mol (assume constant). Compute K2 at 308 K. Output only a number rounded to 2 decimals.",'expected_output':'0.63'},\n  {'id':'Q2','input':"Van't Hoff osmotic pressure: i=2, M=0.15 mol/L, T=298 K, R=0.082057 L·atm·mol^-1·K^-1. Compute π in atm. Output only a number rounded to 2 decimals.",'expected_output':'7.34'},\n  {'id':'Q3','input':'Biostatistics: Treatment event rate is 12/150; control event rate is 30/150. Compute relative risk (treatment/control). Output only a number rounded to 2 decimals.','expected_output':'0.40'},\n  {'id':'Q4','input':'Biostatistics Bayes task: prevalence=0.02, sensitivity=0.90, specificity=0.95. Compute positive predictive value. Output only a number rounded to 2 decimals.','expected_output':'0.27'},\n  {'id':'Q5','input':'Given TP=42, FP=18, FN=8, compute F1 = 2TP/(2TP+FP+FN). Output only a number rounded to 3 decimals.','expected_output':'0.764'},\n  {'id':'Q6','input':"Combinatorial game (Nim variant): one heap has 17 stones, each move removes 1 to 3 stones, normal play. What is the optimal first move? Output only 'remove 1', 'remove 2', or 'remove 3'.",'expected_output':'remove 1'},\n  {'id':'Q7','input':'Normal-play Nim with heaps (3,4,5). Give one winning first-move resulting heap triple in parentheses with commas and no spaces. Output only the triple.','expected_output':'(1,4,5)'},\n  {'id':'Q8','input':'Explain in at most 12 words why xor-sum 0 is losing in normal-play Nim. Output one sentence only.','expected_output':'Any move makes xor nonzero, so the opponent can restore zero.'},\n  {'id':'Q9','input':'Statistics: If probability p=0.8, compute log-odds ln(p/(1-p)). Output only a number rounded to 3 decimals.','expected_output':'1.386'},\n  {'id':'Q10','input':'Hardy-Weinberg: if recessive allele frequency q=0.20, what is carrier frequency 2pq? Output only a number rounded to 2 decimals.','expected_output':'0.32'},\n]

In [ ]:
rows = []\nfor p in prompts:\n    prompt = 'You are a precise assistant. Return only the final answer in the requested format.\nUser: ' + p['input'] + '\nAssistant:'\n    inputs = tok(prompt, return_tensors='pt').to(device)\n    with torch.no_grad():\n        out = model.generate(**inputs, do_sample=False, max_new_tokens=80, pad_token_id=tok.eos_token_id)\n    text = tok.decode(out[0], skip_special_tokens=True)\n    completion = text[len(prompt):].strip() if text.startswith(prompt) else text.strip()\n    rows.append({'id': p['id'], 'input': p['input'], 'expected_output': p['expected_output'], 'actual_output': completion})\n\nrows[:2]